# Density-window 4 km SIF training
Three-level U-Net with 20 predictor channels, equal-footprint aggregate supervision, a fixed SIF-date-grouped 80/10/10 split, and three training seeds.

This notebook uses only the prepared NPZ shards and their companion CSV/JSON files. It does not group by Sentinel source observations, perform cross-validation, or evaluate unseen tiles. The internal test measures held-out SIF dates within the sampled regions; Sentinel acquisition independence is not enforced.

All statistics are fitted on the training partition. Validation selects checkpoints and fits calibration. Test results must not be used to choose configurations. Fine-scale maps are predictions constrained by aggregate labels, not validated 20 m measurements.


## 1. Imports and configuration
Attach the prepared density dataset to Kaggle. Automatic discovery requires exactly one matching directory; otherwise set CHIP_DIR explicitly. Change RUN_NAME for a different experiment. Completed seed runs can be reused only with matching data, split and training settings.

Start with SIMILARITY_LAMBDA = 0.0 for the original aggregate objective. For a later experiment, change SIMILARITY_LAMBDA (for example, 0.01 as an initial trial, not an established optimum) and RUN_NAME. Keep the dataset, SPLIT_SEED, split settings and TRAINING_SEEDS fixed. Select lambda using validation performance and map diagnostics, without tuning on test results. No NPZ or preparation-script changes are needed.


In [ ]:
from pathlib import Path
from collections import OrderedDict
import gc
import hashlib
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import Transformer
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from IPython.display import display

CHIP_DIR = None  # Example: Path('/kaggle/input/your-dataset/spatial_aggregate_density_4km_20m_s2valid95')
DATASET_DIRECTORY_NAME = 'spatial_aggregate_density_4km_20m_s2valid95'
EXPECTED_SAMPLE_COUNT = 13929  # Set to None if intentionally using a different complete dataset.
RUN_NAME = 'density_4km_unet_v1'
OUTPUT_DIR = Path('/kaggle/working') / RUN_NAME

SPLIT_SEED = 42
SPLIT_FRACTIONS = {'train': 0.80, 'validation': 0.10, 'test': 0.10}
SPLIT_RESTARTS = 128
TRAINING_SEEDS = [42, 123, 2026]
BASE_CHANNELS = 16
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
MAX_EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HUBER_BETA = 1.0  # In standardized target units.
SIMILARITY_LAMBDA = 0.0  # Zero preserves the original training objective.
SIMILARITY_SPECTRAL_CHANNELS = ['ndmi', 'ndvi', 'evi', 'nirv', 'ndre']
SIMILARITY_CROP_CHANNELS = [
    'winter_wheat_fraction', 'winter_barley_fraction', 'winter_rye_fraction',
    'maize_fraction', 'grass_forage_fraction', 'woody_fraction',
    'other_crop_fraction', 'non_crop_fraction',
]
SIMILARITY_SPECTRAL_SIGMA = 0.5  # RMS index difference in training-standardized units.
SIMILARITY_CROP_SIGMA = 0.15  # Half the L1 difference of raw crop-fraction vectors.
SIMILARITY_SUPPORT_ONLY = True  # Both pixels must fall within aggregate footprint support.
if not np.isfinite(SIMILARITY_LAMBDA) or SIMILARITY_LAMBDA < 0:
    raise ValueError('SIMILARITY_LAMBDA must be finite and non-negative.')
if any(not np.isfinite(sigma) or sigma <= 0 for sigma in
       (SIMILARITY_SPECTRAL_SIGMA, SIMILARITY_CROP_SIGMA)):
    raise ValueError('Similarity bandwidths must be finite and positive.')
EARLY_STOPPING_PATIENCE = 12
LR_PATIENCE = 4
LR_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-6
GRADIENT_CLIP_NORM = 1.0
USE_AUGMENTATION = True
USE_AMP = True
NUM_WORKERS = 2
SHARD_CACHE_SIZE = 2  # Per dataset/worker; shards store float16 predictors.
NORMALIZATION_STD_FLOOR = 1e-6
MIN_FOOTPRINTS = 4
MIN_GROUP_N = 20
MIN_GROUP_DATES = 3
PERMUTATION_REPEATS = 3
RUN_PERMUTATION_IMPORTANCE = True
REUSE_COMPLETED_RUNS = True

EXPECTED_CHANNELS = [
    'ndmi', 'ndvi', 'evi', 'nirv', 'ndre', 'fapar', 'par', 'apar', 'nirvp',
    'winter_wheat_fraction', 'winter_barley_fraction', 'winter_rye_fraction',
    'maize_fraction', 'grass_forage_fraction', 'woody_fraction', 'other_crop_fraction',
    'active_crop_fraction', 'non_crop_fraction', 'doy_sin', 'doy_cos',
]
TARGET_NAME = 'aggregated_target_modis_sif'
CHIP_SHAPE = (20, 200, 200)
BALANCE_COLUMNS = [
    'month', 'sif_year', 'measurement_mode', 'mgrs_tile_t', 'land_cover_label'
]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = USE_AMP and DEVICE.type == 'cuda'

if CHIP_DIR is None:
    matches = sorted({
        p.parent for p in Path('/kaggle/input').rglob('chip_metadata.csv')
        if p.parent.name == DATASET_DIRECTORY_NAME
    })
    if len(matches) != 1:
        raise ValueError(f'Set CHIP_DIR explicitly; found {len(matches)} matching datasets: {matches}')
    CHIP_DIR = matches[0]
CHIP_DIR = Path(CHIP_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)

def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

def save_figure(fig, name):
    fig.savefig(FIGURE_DIR / f'{name}.png', dpi=160, bbox_inches='tight')
    fig.savefig(FIGURE_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 80)
print('Dataset:', CHIP_DIR)
print('Output:', OUTPUT_DIR)
print('Device:', DEVICE, '| mixed precision:', AMP_ENABLED)
print('Effective batch size:', BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)


## 2. Load shards and metadata
The audit reads one shard at a time. It checks the actual arrays and links rows by aggregation_id rather than relying on CSV order. No target-range clipping is applied.


In [ ]:
metadata_path = CHIP_DIR / 'chip_metadata.csv'
footprint_path = CHIP_DIR / 'footprint_metadata.csv'
config_path = CHIP_DIR / 'dataset_config.json'
metadata = pd.read_csv(metadata_path)
footprint_metadata = pd.read_csv(footprint_path)
dataset_config = json.loads(config_path.read_text(encoding='utf-8'))

required = [
    'aggregation_id', 'Delta_Date', 'sif_year', 'sif_month', 'sif_doy',
    'measurement_mode', 'mgrs_tile_t', 'window_crs', 'y_aggregate', 'n_footprints',
    'cell_xmin', 'cell_ymin', 'cell_xmax', 'cell_ymax',
    'aggregate_support_fraction', 'mean_rasterized_mask_inside_fraction',
    'target_sd', 'target_se',
]
missing = sorted(set(required) - set(metadata.columns))
if missing:
    raise ValueError(f'Missing density metadata columns: {missing}')
metadata['aggregation_id'] = metadata['aggregation_id'].astype(str)
metadata['Delta_Date'] = pd.to_datetime(metadata['Delta_Date'], errors='raise').dt.normalize()
metadata['month'] = metadata['Delta_Date'].dt.month
for column in ['sif_year', 'sif_month', 'sif_doy', 'measurement_mode', 'n_footprints', 'y_aggregate']:
    metadata[column] = pd.to_numeric(metadata[column], errors='raise')
if metadata['aggregation_id'].duplicated().any() or metadata['Delta_Date'].isna().any():
    raise ValueError('Duplicate aggregation IDs or missing SIF dates.')
for column, derived in [
    ('sif_year', metadata['Delta_Date'].dt.year),
    ('sif_month', metadata['month']),
    ('sif_doy', metadata['Delta_Date'].dt.dayofyear),
]:
    if not np.array_equal(metadata[column].to_numpy(), derived.to_numpy()):
        raise ValueError(f'{column} disagrees with Delta_Date.')
land_cover = metadata.get('majority_land_cover_class', pd.Series('', index=metadata.index))
land_cover = land_cover.fillna('').astype(str).str.strip()
fallback = metadata.get('majority_land_cover', pd.Series('(missing)', index=metadata.index))
metadata['land_cover_label'] = land_cover.where(land_cover.ne(''), fallback).fillna('(missing)').astype(str)
if dataset_config.get('channels') != EXPECTED_CHANNELS:
    raise ValueError('Dataset config is not the expected 20-channel DOY/NIRvP schema.')
if dataset_config.get('target') != TARGET_NAME:
    raise ValueError('Unexpected dataset target.')

def decode_strings(values):
    return [v.decode('utf-8') if isinstance(v, bytes) else str(v) for v in values]

shard_paths = sorted(CHIP_DIR.glob('chips_*.npz'))
if not shard_paths:
    raise FileNotFoundError(f'No chips_*.npz files in {CHIP_DIR}')
index_rows = []
finite_totals = np.zeros(len(EXPECTED_CHANNELS), dtype=np.int64)
total_pixels = 0
shard_signatures = []
for shard_number, path in enumerate(shard_paths):
    stat = path.stat()
    shard_signatures.append((path.name, stat.st_size, stat.st_mtime_ns))
    with np.load(path, allow_pickle=False) as shard:
        needed = {'X', 'aggregate_weight_map', 'y_aggregate', 'n_footprints',
                  'aggregation_id', 'channel_names', 'target_name'}
        if not needed.issubset(shard.files):
            raise ValueError(f'{path.name}: missing keys {needed - set(shard.files)}')
        if decode_strings(shard['channel_names']) != EXPECTED_CHANNELS:
            raise ValueError(f'{path.name}: channel order mismatch')
        if decode_strings(shard['target_name']) != [TARGET_NAME]:
            raise ValueError(f'{path.name}: target mismatch')
        ids = decode_strings(shard['aggregation_id'])
        x = shard['X']
        weight = shard['aggregate_weight_map'].astype(np.float32)
        y = shard['y_aggregate']
        counts = shard['n_footprints']
        n = len(ids)
        if x.shape != (n, *CHIP_SHAPE) or weight.shape != (n, 200, 200):
            raise ValueError(f'{path.name}: unexpected predictor/weight shape')
        if y.shape != (n,) or counts.shape != (n,):
            raise ValueError(f'{path.name}: unexpected target/count shape')
        if not np.isfinite(y).all() or not np.isfinite(counts).all() or (counts < MIN_FOOTPRINTS).any():
            raise ValueError(f'{path.name}: invalid targets or footprint counts')
        if not np.isfinite(weight).all() or (weight < 0).any():
            raise ValueError(f'{path.name}: non-finite or negative weights')
        weight_sums = weight.sum(axis=(1, 2), dtype=np.float64)
        if not np.allclose(weight_sums, 1.0, atol=0.01, rtol=0):
            raise ValueError(f'{path.name}: stored weight sums are not near one')
        if np.isinf(x).any():
            raise ValueError(f'{path.name}: infinite predictors')
        finite_per_sample = np.isfinite(x).sum(axis=(2, 3))
        if (finite_per_sample == 0).any():
            raise ValueError(f'{path.name}: a sample has a completely missing channel')
        finite_totals += finite_per_sample.sum(axis=0)
        total_pixels += n * 200 * 200
        for local_index, aggregation_id in enumerate(ids):
            index_rows.append({
                'aggregation_id': aggregation_id, 'shard_path': str(path),
                'local_index': local_index, 'npz_target': float(y[local_index]),
                'npz_n_footprints': int(counts[local_index]),
                'stored_weight_sum': float(weight_sums[local_index]),
            })
    if (shard_number + 1) % 50 == 0 or shard_number + 1 == len(shard_paths):
        print(f'Audited {shard_number + 1}/{len(shard_paths)} shards')

shard_index = pd.DataFrame(index_rows)
if shard_index['aggregation_id'].duplicated().any():
    raise ValueError('Duplicate aggregation IDs across shards; check for stale NPZ files.')
if set(metadata['aggregation_id']) != set(shard_index['aggregation_id']):
    raise ValueError('Metadata and NPZ aggregation IDs do not match exactly.')
samples = metadata.merge(shard_index, on='aggregation_id', validate='one_to_one')
if not np.allclose(samples['y_aggregate'], samples['npz_target'], atol=1e-7, rtol=1e-6):
    raise ValueError('Metadata targets disagree with stored NPZ targets.')
if not np.array_equal(samples['n_footprints'], samples['npz_n_footprints']):
    raise ValueError('Metadata footprint counts disagree with NPZ counts.')
samples = samples.sort_values(['Delta_Date', 'mgrs_tile_t', 'aggregation_id']).reset_index(drop=True)
if EXPECTED_SAMPLE_COUNT is not None and len(samples) != EXPECTED_SAMPLE_COUNT:
    raise ValueError(f'Expected {EXPECTED_SAMPLE_COUNT} windows, found {len(samples)}.')

footprint_metadata['aggregation_id'] = footprint_metadata['aggregation_id'].astype(str)
if not {'observed_sif', 'slot'}.issubset(footprint_metadata.columns):
    raise ValueError('Footprint metadata must contain observed_sif and slot.')
if footprint_metadata.duplicated(['aggregation_id', 'slot']).any():
    raise ValueError('Duplicate footprint slots within an aggregate.')
if set(footprint_metadata['aggregation_id']) != set(samples['aggregation_id']):
    raise ValueError('Footprint metadata and chip IDs do not match.')
footprint_metadata['observed_sif'] = pd.to_numeric(footprint_metadata['observed_sif'], errors='raise')
if not np.isfinite(footprint_metadata['observed_sif']).all():
    raise ValueError('Non-finite footprint observations.')
footprint_check = footprint_metadata.groupby('aggregation_id')['observed_sif'].agg(['count', 'mean'])
aligned = footprint_check.reindex(samples['aggregation_id'])
if not np.array_equal(aligned['count'].to_numpy(), samples['n_footprints'].to_numpy()):
    raise ValueError('Footprint table counts do not match the chip metadata.')
if not np.allclose(aligned['mean'], samples['y_aggregate'], atol=1e-7, rtol=1e-6):
    raise ValueError('Footprint means do not match aggregate targets.')

channel_names = EXPECTED_CHANNELS.copy()
DATASET_FINGERPRINT = fingerprint({
    'shards': shard_signatures,
    'metadata_sha256': hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
    'footprints_sha256': hashlib.sha256(footprint_path.read_bytes()).hexdigest(),
    'dataset_config': dataset_config,
})
# Dataset-wide counts are descriptive only; they do not set normalization.
input_audit = pd.DataFrame({
    'channel': channel_names, 'finite_pixels': finite_totals,
    'total_pixels': total_pixels, 'missing_fraction': 1 - finite_totals / total_pixels,
})
input_audit.to_csv(OUTPUT_DIR / 'input_channel_audit.csv', index=False)
del x, weight, finite_per_sample
gc.collect()
print(f'{len(samples):,} windows across {samples.Delta_Date.nunique():,} SIF dates')
display(samples[['y_aggregate', 'n_footprints', 'target_sd', 'target_se']].describe())
display(input_audit)


## 3. Dataset overview
These are descriptive sample distributions. Negative and high accepted SIF targets are retained. Land-cover labels describe the selected footprint class; they do not imply every pixel in a window has that class.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(samples['y_aggregate'], bins=50)
axes[0].set(xlabel='Aggregate observed SIF', ylabel='Windows', title='Target distribution')
axes[1].hist(samples['n_footprints'], bins=30)
axes[1].set(xlabel='Contributing footprints', ylabel='Windows', title='Supervision counts')
monthly_counts = samples.groupby('month').size()
axes[2].bar(monthly_counts.index.astype(str), monthly_counts.values)
axes[2].set(xlabel='SIF month', ylabel='Windows', title='Seasonal sampling')
save_figure(fig, 'dataset_overview')
for column in BALANCE_COLUMNS:
    overview = samples.groupby(column, dropna=False).agg(
        n_windows=('aggregation_id', 'size'), n_dates=('Delta_Date', 'nunique')
    ).reset_index()
    overview.to_csv(OUTPUT_DIR / f'dataset_counts_{column}.csv', index=False)
    display(overview)


## 4. Fixed SIF-date-grouped split
Every Delta_Date is indivisible across all tiles and modes. A seeded search approximately balances window count and the marginal distributions of month, year, mode, tile and land cover. Rare categories are soft preferences, not hard requirements. No SIF targets or model scores are used to optimize the allocation.

The saved split is reused unchanged across all training seeds. No Sentinel-source or spatial-overlap grouping is performed.


In [ ]:
split_path = OUTPUT_DIR / 'chip_splits.csv'
split_config_path = OUTPUT_DIR / 'split_config.json'
SPLIT_SETTINGS = {
    'dataset_fingerprint': DATASET_FINGERPRINT, 'seed': SPLIT_SEED,
    'fractions': SPLIT_FRACTIONS, 'balance_columns': BALANCE_COLUMNS,
    'restarts': SPLIT_RESTARTS, 'group_column': 'Delta_Date', 'algorithm_version': 1,
}
SPLIT_SIGNATURE = fingerprint(SPLIT_SETTINGS)
split_names = list(SPLIT_FRACTIONS)
fractions = np.array(list(SPLIT_FRACTIONS.values()), dtype=np.float64)
if not np.isclose(fractions.sum(), 1.0) or (fractions <= 0).any():
    raise ValueError('Split fractions must be positive and sum to one.')

def allocate_dates(table):
    dates = pd.Index(sorted(table['Delta_Date'].unique()), name='Delta_Date')
    if len(dates) < 3:
        raise ValueError('At least three distinct SIF dates are needed.')
    date_sizes = table.groupby('Delta_Date').size().reindex(dates).to_numpy(dtype=np.float64)
    pieces = [date_sizes[:, None]]
    weights = [4.0]  # Total sample balance is emphasized.
    for column in BALANCE_COLUMNS:
        labels = table[column].fillna('(missing)').astype(str)
        counts = pd.crosstab(table['Delta_Date'], labels).reindex(dates, fill_value=0)
        pieces.append(counts.to_numpy(dtype=np.float64))
        weights.extend([1.0 / counts.shape[1]] * counts.shape[1])
    vectors = np.concatenate(pieces, axis=1)
    weights = np.asarray(weights)
    target = fractions[:, None] * vectors.sum(axis=0)[None, :]
    scale = np.maximum(target, 1.0)

    def objective(counts):
        return float(np.sum(weights[None, :] * ((counts - target) / scale) ** 2))

    rng = np.random.default_rng(SPLIT_SEED)
    best_score, best_assignment, best_counts = np.inf, None, None
    for _ in range(SPLIT_RESTARTS):
        order = np.argsort(-(date_sizes * rng.uniform(0.75, 1.25, len(dates))))
        assignment = np.full(len(dates), -1, dtype=int)
        counts = np.zeros_like(target)
        for group_index in order:
            scores = []
            for partition in range(3):
                proposal = counts.copy()
                proposal[partition] += vectors[group_index]
                scores.append(objective(proposal))
            scores = np.asarray(scores)
            candidates = np.flatnonzero(np.isclose(scores, scores.min(), rtol=1e-12, atol=1e-12))
            partition = int(rng.choice(candidates))
            assignment[group_index] = partition
            counts[partition] += vectors[group_index]
        if len(np.unique(assignment)) < 3:
            continue
        score = objective(counts)
        if score < best_score:
            best_score, best_assignment, best_counts = score, assignment.copy(), counts.copy()
    if best_assignment is None:
        raise RuntimeError('Could not form three nonempty grouped partitions.')

    # Refine by moving whole dates only; no partition may become empty.
    groups_per_split = np.bincount(best_assignment, minlength=3)
    for _ in range(10):
        improved = False
        for group_index in rng.permutation(len(dates)):
            source = int(best_assignment[group_index])
            if groups_per_split[source] <= 1:
                continue
            chosen, chosen_score, chosen_counts = source, best_score, None
            for destination in range(3):
                if destination == source:
                    continue
                proposal = best_counts.copy()
                proposal[source] -= vectors[group_index]
                proposal[destination] += vectors[group_index]
                score = objective(proposal)
                if score < chosen_score - 1e-12:
                    chosen, chosen_score, chosen_counts = destination, score, proposal
            if chosen != source:
                best_assignment[group_index] = chosen
                groups_per_split[source] -= 1
                groups_per_split[chosen] += 1
                best_score, best_counts = chosen_score, chosen_counts
                improved = True
        if not improved:
            break
    print('Split balance objective:', best_score)
    return pd.Series([split_names[i] for i in best_assignment], index=dates)

if split_path.exists() or split_config_path.exists():
    if not (split_path.exists() and split_config_path.exists()):
        raise RuntimeError('Incomplete saved split. Use a new RUN_NAME or restore both split files.')
    saved_settings = json.loads(split_config_path.read_text())
    if saved_settings.get('signature') != SPLIT_SIGNATURE:
        raise ValueError('Saved split belongs to different data/settings. Choose a new RUN_NAME.')
    saved_split = pd.read_csv(split_path, dtype={'aggregation_id': str}, parse_dates=['Delta_Date'])
    if saved_split['aggregation_id'].duplicated().any() or set(saved_split.aggregation_id) != set(samples.aggregation_id):
        raise ValueError('Saved split IDs do not match current data.')
    samples = samples.merge(
        saved_split[['aggregation_id', 'Delta_Date', 'split']],
        on=['aggregation_id', 'Delta_Date'], how='left', validate='one_to_one'
    )
    print('Reused saved date allocation.')
else:
    date_assignment = allocate_dates(samples)
    samples['split'] = samples['Delta_Date'].map(date_assignment)
    samples[['aggregation_id', 'Delta_Date', 'split']].to_csv(split_path, index=False)
    save_json(split_config_path, {'signature': SPLIT_SIGNATURE, **SPLIT_SETTINGS})

if samples['split'].isna().any() or not samples['split'].isin(split_names).all():
    raise ValueError('Invalid or missing split labels.')
if samples.groupby('Delta_Date')['split'].nunique().max() != 1:
    raise ValueError('A SIF date crosses partitions.')
tables = {
    name: samples.loc[samples['split'].eq(name)].reset_index(drop=True)
    for name in split_names
}
if any(table.empty for table in tables.values()):
    raise ValueError('An empty partition was produced.')
train_table, val_table, test_table = (tables[name] for name in split_names)
date_assignments = samples.groupby('Delta_Date').agg(
    split=('split', 'first'), n_windows=('aggregation_id', 'size')
).reset_index()
date_assignments.to_csv(OUTPUT_DIR / 'date_splits.csv', index=False)
split_summary = samples.groupby('split').agg(
    n_windows=('aggregation_id', 'size'), n_dates=('Delta_Date', 'nunique'),
    n_tiles=('mgrs_tile_t', 'nunique')
).reindex(split_names)
split_summary['window_fraction'] = split_summary['n_windows'] / len(samples)
split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv')
display(split_summary)

for column in BALANCE_COLUMNS:
    balance_table = samples.assign(category=samples[column].fillna('(missing)').astype(str))
    counts = pd.crosstab(balance_table['category'], balance_table['split']).reindex(columns=split_names, fill_value=0)
    counts.to_csv(OUTPUT_DIR / f'split_balance_{column}.csv')
    proportions = counts.div(counts.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(7, max(3, min(16, 0.35 * len(counts) + 1))))
    sns.heatmap(proportions, annot=counts, fmt='d', vmin=0, vmax=1, cmap='Blues', ax=ax)
    ax.set(title=f'{column}: fraction of each category; annotations = windows', xlabel='Partition')
    save_figure(fig, f'split_balance_{column}')
    missing_levels = counts.index[(counts == 0).any(axis=1)].tolist()
    if missing_levels:
        print(f'{column}: categories absent from at least one partition:', missing_levels)

fig, ax = plt.subplots(figsize=(8, 4))
target_edges = np.histogram_bin_edges(samples['y_aggregate'], bins=40)
for name, table in tables.items():
    ax.hist(table['y_aggregate'], bins=target_edges, density=True, histtype='step', label=name)
ax.set(xlabel='Observed aggregate SIF', ylabel='Density', title='Target distribution by fixed partition')
ax.legend()
save_figure(fig, 'split_target_distributions')


## 5. Normalization from every training chip
Predictor means and variances include all finite pixels in every training chip, with equal weight per represented pixel. Overlapping imagery is counted as represented in each training sample. Float64 moment accumulation avoids float16 overflow and improves numerical stability.

NaNs remain missing during estimation and become zero only after normalization. Constant training channels use scale 1 instead of an extremely small divisor. Target mean and standard deviation use all training targets. No validation or test values contribute.


In [ ]:
stats_path = OUTPUT_DIR / 'normalization_stats.npz'
NORMALIZATION_SIGNATURE = fingerprint({
    'dataset': DATASET_FINGERPRINT, 'split': SPLIT_SIGNATURE,
    'train_ids': sorted(train_table['aggregation_id'].tolist()),
    'channels': channel_names, 'std_floor': NORMALIZATION_STD_FLOOR,
    'method': 'all-training-finite-pixels-float64-merged-moments-v1',
})
if stats_path.exists():
    with np.load(stats_path, allow_pickle=False) as stats:
        if str(stats['signature'].item()) != NORMALIZATION_SIGNATURE:
            raise ValueError('Normalization cache mismatch. Use a new RUN_NAME.')
        channel_mean = stats['mean'].copy()
        channel_std = stats['scale'].copy()
        raw_channel_std = stats['raw_std'].copy()
        channel_count = stats['count'].copy()
        channel_min = stats['minimum'].copy()
        channel_max = stats['maximum'].copy()
        target_mean = float(stats['target_mean'])
        target_std = float(stats['target_std'])
    print('Reused matching all-training normalization statistics.')
else:
    c = len(channel_names)
    channel_count = np.zeros(c, dtype=np.int64)
    running_mean = np.zeros(c, dtype=np.float64)
    running_m2 = np.zeros(c, dtype=np.float64)
    channel_min = np.full(c, np.inf)
    channel_max = np.full(c, -np.inf)
    processed = 0
    for path, rows in train_table.groupby('shard_path', sort=False):
        with np.load(path, allow_pickle=False) as shard:
            shard_x = shard['X']
            for local_index in rows['local_index'].to_numpy(dtype=int):
                array = shard_x[local_index].astype(np.float64)
                finite = np.isfinite(array)
                counts = finite.sum(axis=(1, 2))
                if (counts == 0).any():
                    raise ValueError('Training sample has a completely missing channel.')
                means = np.where(finite, array, 0).sum(axis=(1, 2)) / counts
                centered = np.where(finite, array - means[:, None, None], 0)
                m2 = np.square(centered).sum(axis=(1, 2))
                combined = channel_count + counts
                delta = means - running_mean
                running_m2 += m2 + delta ** 2 * channel_count * counts / combined
                running_mean += delta * counts / combined
                channel_count = combined
                channel_min = np.minimum(channel_min, np.where(finite, array, np.inf).min(axis=(1, 2)))
                channel_max = np.maximum(channel_max, np.where(finite, array, -np.inf).max(axis=(1, 2)))
                processed += 1
        if processed % 256 < len(rows) or processed == len(train_table):
            print(f'Normalization: {processed:,}/{len(train_table):,} training chips')
    raw_channel_std = np.sqrt(np.maximum(running_m2 / channel_count, 0))
    channel_mean = running_mean.astype(np.float32)
    channel_std = np.where(raw_channel_std < NORMALIZATION_STD_FLOOR, 1.0, raw_channel_std).astype(np.float32)
    target_mean = float(train_table['y_aggregate'].mean())
    target_std = float(train_table['y_aggregate'].std(ddof=0))
    if not np.isfinite(target_std) or target_std <= 0:
        raise ValueError('Training targets have no usable variance.')
    np.savez_compressed(
        stats_path, signature=np.asarray(NORMALIZATION_SIGNATURE),
        channel_names=np.asarray(channel_names), mean=channel_mean, scale=channel_std,
        raw_std=raw_channel_std, count=channel_count, minimum=channel_min, maximum=channel_max,
        target_mean=np.asarray(target_mean), target_std=np.asarray(target_std),
    )
    del shard_x, array, finite, centered
    gc.collect()

normalization_table = pd.DataFrame({
    'channel': channel_names, 'mean': channel_mean, 'raw_std': raw_channel_std,
    'normalization_scale': channel_std, 'minimum': channel_min, 'maximum': channel_max,
    'finite_pixels': channel_count,
    'missing_fraction': 1 - channel_count / (len(train_table) * 200 * 200),
    'constant_or_nearly_constant': raw_channel_std < NORMALIZATION_STD_FLOOR,
})
normalization_table.to_csv(OUTPUT_DIR / 'training_channel_statistics.csv', index=False)
display(normalization_table)
print('Training target mean / std:', target_mean, target_std)


## 6. Dataset, shard cache and joint augmentation
Flips and 90-degree rotations transform predictors, supervision weights and similarity-validity masks together; targets are unchanged. The training dataset preserves finite-value validity for the similarity channels before missing predictors are replaced with normalized zero. This cannot identify Sentinel pixels already filled upstream: original cloud-validity masks are not stored in these NPZ files. Validation, test and training-evaluation datasets are never augmented and retain the original four-item return format. Batches pack across shard boundaries to avoid many tiny batches while limiting repeated NPZ decompression.


In [ ]:
class ShardCache:
    def __init__(self, capacity=2):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, path):
        if path in self.cache:
            self.cache.move_to_end(path)
            return self.cache[path]
        with np.load(path, allow_pickle=False) as shard:
            arrays = {key: shard[key] for key in ['X', 'aggregate_weight_map', 'y_aggregate']}
        self.cache[path] = arrays
        while len(self.cache) > self.capacity:
            self.cache.popitem(last=False)
        return arrays

SIMILARITY_SPECTRAL_INDICES = [channel_names.index(name) for name in SIMILARITY_SPECTRAL_CHANNELS]
SIMILARITY_CROP_INDICES = [channel_names.index(name) for name in SIMILARITY_CROP_CHANNELS]
SIMILARITY_VALID_INDICES = sorted(set(SIMILARITY_SPECTRAL_INDICES + SIMILARITY_CROP_INDICES))
if not SIMILARITY_SPECTRAL_INDICES or not SIMILARITY_CROP_INDICES:
    raise ValueError('Similarity requires non-empty spectral and crop channel lists.')

class DensityDataset(Dataset):
    def __init__(self, table, augment=False, return_similarity_valid=False):
        self.table = table.reset_index(drop=True).copy()
        self.augment = augment
        self.return_similarity_valid = return_similarity_valid
        self.cache = ShardCache(SHARD_CACHE_SIZE)

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        shard = self.cache.get(str(row['shard_path']))
        local_index = int(row['local_index'])
        x = shard['X'][local_index].astype(np.float32, copy=True)
        similarity_valid = (np.isfinite(x[SIMILARITY_VALID_INDICES]).all(axis=0)
                            if self.return_similarity_valid else None)
        weight = shard['aggregate_weight_map'][local_index].astype(np.float32, copy=True)
        y = np.float32((float(shard['y_aggregate'][local_index]) - target_mean) / target_std)
        weight_sum = weight.sum(dtype=np.float64)
        if not np.isfinite(weight_sum) or weight_sum <= 0:
            raise ValueError(f'Invalid weight map: {row.aggregation_id}')
        weight /= weight_sum
        x = (x - channel_mean[:, None, None]) / channel_std[:, None, None]
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        if self.augment:
            rotations = np.random.randint(4)
            x = np.rot90(x, rotations, axes=(1, 2))
            weight = np.rot90(weight, rotations, axes=(0, 1))
            if similarity_valid is not None:
                similarity_valid = np.rot90(similarity_valid, rotations, axes=(0, 1))
            if np.random.random() < 0.5:
                x, weight = x[:, :, ::-1], weight[:, ::-1]
                if similarity_valid is not None:
                    similarity_valid = similarity_valid[:, ::-1]
            if np.random.random() < 0.5:
                x, weight = x[:, ::-1, :], weight[::-1, :]
                if similarity_valid is not None:
                    similarity_valid = similarity_valid[::-1, :]
        sample = (
            torch.from_numpy(np.ascontiguousarray(x)),
            torch.from_numpy(np.ascontiguousarray(weight)),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(index, dtype=torch.long),
        )
        if similarity_valid is not None:
            sample += (torch.from_numpy(np.ascontiguousarray(similarity_valid)),)
        return sample

class PackedShardBatchSampler(Sampler):
    def __init__(self, table, shuffle, seed):
        self.groups = [
            np.asarray(indices, dtype=np.int64)
            for indices in table.reset_index(drop=True).groupby('shard_path', sort=False).indices.values()
        ]
        self.n = len(table)
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        group_order = np.arange(len(self.groups))
        if self.shuffle:
            rng.shuffle(group_order)
        batch = []
        for group_index in group_order:
            indices = self.groups[group_index].copy()
            if self.shuffle:
                rng.shuffle(indices)
            for index in indices:
                batch.append(int(index))
                if len(batch) == BATCH_SIZE:
                    yield batch
                    batch = []
        if batch:
            yield batch

    def __len__(self):
        return math.ceil(self.n / BATCH_SIZE)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(table, seed, training=False):
    dataset = DensityDataset(table, augment=training and USE_AUGMENTATION,
                             return_similarity_valid=training)
    sampler = PackedShardBatchSampler(table, shuffle=training, seed=seed)
    kwargs = dict(
        batch_sampler=sampler, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(seed),
    )
    if NUM_WORKERS > 0:
        kwargs.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(dataset, **kwargs)

def cleanup_memory():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


## 7. Three-level U-Net, aggregate objective and optional similarity penalty
The model predicts an unconstrained 200 × 200 SIF map. Fractional footprint weights reduce this map to the supervised scalar and are not predictor channels. Smooth L1 is calculated in standardized SIF units. Evaluation returns to the original target units.

The optional penalty compares horizontal and vertical neighbours within each chip (each edge once). Let d_s squared be the mean squared difference of the five training-standardized vegetation indices, and d_c be half the summed absolute difference of the eight original crop/non-crop fractions. Pair affinity is exp[-0.5 * (d_s squared / spectral_sigma squared + d_c squared / crop_sigma squared)]. Similar pairs contribute more; differences across spectral or crop boundaries reduce the penalty. PAR, FAPAR, NIRvP and seasonality are not added to this distance, avoiding repeated weighting of derived or spatially coarse predictors.

Per-chip similarity loss = sum(affinity * squared predicted-SIF difference) / number of eligible neighbour pairs. Both pixels need finite guidance channels; by default both must also have positive aggregate footprint weight. Zero eligible pairs gives zero loss. Dividing by the eligible count, rather than the sum of affinities, prevents weak affinities from being amplified in heterogeneous chips. Crops use their original fraction scale; map differences use standardized SIF units. Affinities are fixed from inputs and receive no gradients.

Training objective = mean over chips of [aggregate Smooth L1 + SIMILARITY_LAMBDA * similarity loss]. This is a local, index-and-crop-guided regularizer, not a reproduction of CS-SUNet's sampled non-local reflectance/land-cover loss. It encourages local consistency without guaranteeing real field boundaries or accurate 20 m SIF. At lambda zero, the penalty is computed without gradients for diagnostics only and does not enter backpropagation.


In [ ]:
def group_count(channels):
    return next(groups for groups in (8, 4, 2, 1) if channels % groups == 0)

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(out_channels), out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(out_channels), out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ThreeLevelUNet(nn.Module):
    def __init__(self, in_channels=20, base_channels=16):
        super().__init__()
        b = base_channels
        self.enc1 = ConvBlock(in_channels, b)
        self.enc2 = ConvBlock(b, 2 * b)
        self.enc3 = ConvBlock(2 * b, 4 * b)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(4 * b, 8 * b)
        self.up3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, stride=2)
        self.dec3 = ConvBlock(8 * b, 4 * b)
        self.up2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, stride=2)
        self.dec2 = ConvBlock(4 * b, 2 * b)
        self.up1 = nn.ConvTranspose2d(2 * b, b, 2, stride=2)
        self.dec1 = ConvBlock(2 * b, b)
        self.out = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        z = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(z), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

def aggregate_prediction(pred_map, weights):
    # Accumulate in float32 even when convolution uses mixed precision.
    return (pred_map[:, 0].float() * weights.float()).sum(dim=(-2, -1))

def similarity_penalty_per_chip(pred_map, x, valid, weights):
    # Float32 arithmetic outside autocast; only map differences carry gradients.
    predictions = pred_map[:, 0].float()
    with torch.no_grad():
        spectral = x[:, SIMILARITY_SPECTRAL_INDICES].detach().float()
        crop_mean = torch.as_tensor(channel_mean[SIMILARITY_CROP_INDICES],
                                    dtype=torch.float32, device=x.device)[None, :, None, None]
        crop_std = torch.as_tensor(channel_std[SIMILARITY_CROP_INDICES],
                                   dtype=torch.float32, device=x.device)[None, :, None, None]
        crops = (x[:, SIMILARITY_CROP_INDICES].detach().float() * crop_std + crop_mean).clamp(0, 1)
        eligible_pixels = valid.bool()
        if SIMILARITY_SUPPORT_ONLY:
            eligible_pixels = eligible_pixels & (weights > 0)
    numerator = predictions.new_zeros(predictions.shape[0])
    pair_count = predictions.new_zeros(predictions.shape[0])
    affinity_sum = predictions.new_zeros(predictions.shape[0])
    # Ellipsis makes the same spatial slices work for B,H,W and B,C,H,W.
    neighbour_slices = [
        ((Ellipsis, slice(None), slice(None, -1)), (Ellipsis, slice(None), slice(1, None))),
        ((Ellipsis, slice(None, -1), slice(None)), (Ellipsis, slice(1, None), slice(None))),
    ]
    for left, right in neighbour_slices:
        with torch.no_grad():
            eligible = eligible_pixels[left] & eligible_pixels[right]
            spectral_distance_sq = (spectral[left] - spectral[right]).square().mean(dim=1)
            crop_distance = 0.5 * (crops[left] - crops[right]).abs().sum(dim=1)
            affinity = torch.exp(-0.5 * (
                spectral_distance_sq / SIMILARITY_SPECTRAL_SIGMA ** 2
                + (crop_distance / SIMILARITY_CROP_SIGMA).square()
            )) * eligible.float()
            pair_count += eligible.float().sum(dim=(-2, -1))
            affinity_sum += affinity.sum(dim=(-2, -1))
        difference_sq = (predictions[left] - predictions[right]).square()
        numerator = numerator + (affinity * difference_sq).sum(dim=(-2, -1))
    denominator = pair_count.clamp_min(1)
    return numerator / denominator, pair_count, affinity_sum / denominator

def regression_metrics(observed, predicted):
    observed = np.asarray(observed, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    if observed.shape != predicted.shape or not (np.isfinite(observed).all() and np.isfinite(predicted).all()):
        raise ValueError('Metrics require matching, finite observations and predictions.')
    n = len(observed)
    if n == 0:
        return {key: np.nan for key in ['rmse', 'mae', 'bias', 'r2', 'slope', 'intercept', 'pearson_r', 'nrmse']}
    residual = predicted - observed
    centered = observed - observed.mean()
    ss_total = np.square(centered).sum()
    slope = float(np.dot(centered, predicted - predicted.mean()) / ss_total) if ss_total > 0 else np.nan
    observed_std = float(observed.std())
    predicted_std = float(predicted.std())
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return {
        'n': n, 'rmse': rmse, 'mae': float(np.abs(residual).mean()),
        'bias': float(residual.mean()),
        'r2': float(1 - np.square(residual).sum() / ss_total) if ss_total > 0 else np.nan,
        'slope': slope,
        'intercept': float(predicted.mean() - slope * observed.mean()) if ss_total > 0 else np.nan,
        'pearson_r': float(np.corrcoef(observed, predicted)[0, 1]) if n > 1 and observed_std > 0 and predicted_std > 0 else np.nan,
        'nrmse': rmse / observed_std if observed_std > 0 else np.nan,
    }

@torch.no_grad()
def predict_table(model, loader):
    model.eval()
    table = loader.dataset.table
    predictions = np.full(len(table), np.nan, dtype=np.float64)
    seen = np.zeros(len(table), dtype=np.int64)
    for x, weights, _, indices in loader:
        x = x.to(DEVICE, non_blocking=True)
        weights = weights.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            pred_map = model(x)
        predicted = aggregate_prediction(pred_map, weights).cpu().numpy() * target_std + target_mean
        indices = indices.numpy()
        predictions[indices] = predicted
        seen[indices] += 1
    if not np.all(seen == 1) or not np.isfinite(predictions).all():
        raise RuntimeError('Prediction did not produce one finite value per sample.')
    result = table.copy()
    result['observed_sif'] = result['y_aggregate'].astype(float)
    result['predicted_sif_raw'] = predictions
    result['residual_raw'] = predictions - result['observed_sif']
    return result

def model_for_seed(seed):
    seed_everything(seed)
    return ThreeLevelUNet(len(channel_names), BASE_CHANNELS).to(DEVICE)

probe_model = model_for_seed(TRAINING_SEEDS[0])
print(f'Trainable parameters: {sum(p.numel() for p in probe_model.parameters()):,}')
del probe_model
cleanup_memory()


## 8. Train the same configuration with three seeds
Maximum 100 epochs; early stopping after 12 unimproved validation epochs, with learning-rate reduction after the scheduler's patience of 4. All seeds share data partitions and normalization.

Aggregate and similarity losses are reduced per chip, accumulated as sums, and gradients are divided by the actual number of samples in each accumulation group. This also handles a final partial batch/group. Raw validation aggregate RMSE alone selects checkpoints, drives the scheduler and triggers early stopping; regularization is not added to evaluation metrics. Training history logs aggregate loss, unweighted similarity, lambda-weighted similarity, total loss, eligible pair counts and affinities. The lambda-zero run still logs the unweighted penalty without adding gradients. Non-finite values stop the run rather than silently producing invalid results.

A completed run with a matching signature can be reused. An incomplete run requires a new RUN_NAME or explicitly setting REUSE_COMPLETED_RUNS=False to restart that seed. This notebook does not implement mid-epoch resume.


In [ ]:
TRAINING_CONFIG = {
    'architecture': 'ThreeLevelUNet', 'base_channels': BASE_CHANNELS,
    'batch_size': BATCH_SIZE, 'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'max_epochs': MAX_EPOCHS, 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'huber_beta': HUBER_BETA, 'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'lr_patience': LR_PATIENCE, 'lr_factor': LR_FACTOR, 'min_learning_rate': MIN_LEARNING_RATE,
    'gradient_clip_norm': GRADIENT_CLIP_NORM, 'augmentation': USE_AUGMENTATION,
    'amp_enabled': AMP_ENABLED, 'num_workers': NUM_WORKERS,
    'training_seeds': TRAINING_SEEDS, 'code_version': 2,
    'similarity': {
        'lambda': SIMILARITY_LAMBDA, 'spectral_channels': SIMILARITY_SPECTRAL_CHANNELS,
        'crop_channels': SIMILARITY_CROP_CHANNELS,
        'spectral_sigma': SIMILARITY_SPECTRAL_SIGMA, 'crop_sigma': SIMILARITY_CROP_SIGMA,
        'support_only': SIMILARITY_SUPPORT_ONLY, 'neighbours': 'horizontal_vertical',
        'normalization': 'eligible_pair_count_per_chip',
    },
}
RUN_SIGNATURE = fingerprint({
    'training': TRAINING_CONFIG, 'normalization': NORMALIZATION_SIGNATURE,
    'split': SPLIT_SIGNATURE, 'channels': channel_names,
})
experiment_config_path = OUTPUT_DIR / 'experiment_config.json'
if experiment_config_path.exists():
    previous_config = json.loads(experiment_config_path.read_text())
    if previous_config.get('run_signature') != RUN_SIGNATURE:
        raise ValueError('This RUN_NAME already has different training settings. Choose a new RUN_NAME.')
save_json(experiment_config_path, {
    'training': TRAINING_CONFIG, 'dataset_fingerprint': DATASET_FINGERPRINT,
    'split_settings': SPLIT_SETTINGS, 'normalization_signature': NORMALIZATION_SIGNATURE,
    'run_signature': RUN_SIGNATURE, 'channel_names': channel_names,
    'torch_version': str(torch.__version__), 'numpy_version': np.__version__,
    'pandas_version': pd.__version__, 'device': str(DEVICE),
})

def save_checkpoint(path, model, seed, epoch, val_rmse):
    temporary = path.with_name(path.name + '.tmp')
    torch.save({
        'model_state_dict': {name: value.detach().cpu().clone() for name, value in model.state_dict().items()},
        'model_class': 'ThreeLevelUNet', 'base_channels': BASE_CHANNELS,
        'channel_names': channel_names, 'target_name': TARGET_NAME,
        'channel_mean': channel_mean.tolist(), 'channel_std': channel_std.tolist(),
        'target_mean': target_mean, 'target_std': target_std,
        'seed': int(seed), 'best_epoch': int(epoch), 'best_validation_rmse': float(val_rmse),
        'run_signature': RUN_SIGNATURE, 'training_config': TRAINING_CONFIG,
        'dataset_config': dataset_config, 'supervision': 'equal-footprint weighted aggregate',
        'seasonal_encoding': dataset_config.get('seasonal_encoding'),
    }, temporary)
    temporary.replace(path)

def load_seed_model(seed):
    path = OUTPUT_DIR / f'seed_{seed}' / 'best_model.pt'
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    if checkpoint['run_signature'] != RUN_SIGNATURE or checkpoint['channel_names'] != channel_names:
        raise ValueError(f'Checkpoint mismatch for seed {seed}')
    model = model_for_seed(seed)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model, checkpoint

def train_one_seed(seed):
    seed_dir = OUTPUT_DIR / f'seed_{seed}'
    seed_dir.mkdir(exist_ok=True)
    checkpoint_path = seed_dir / 'best_model.pt'
    completed_path = seed_dir / 'training_complete.json'
    if REUSE_COMPLETED_RUNS and completed_path.exists():
        record = json.loads(completed_path.read_text())
        if record['run_signature'] != RUN_SIGNATURE or not checkpoint_path.exists():
            raise ValueError(f'Seed {seed}: completed run has different settings or lacks its checkpoint.')
        print(f'Seed {seed}: reusing completed run.')
        return record
    if REUSE_COMPLETED_RUNS and checkpoint_path.exists():
        raise RuntimeError(f'Seed {seed}: incomplete run. Choose a new RUN_NAME or set REUSE_COMPLETED_RUNS=False to restart.')

    # Invalidate a previous completion marker before an explicitly requested restart.
    if completed_path.exists():
        completed_path.unlink()
    model = model_for_seed(seed)
    train_loader = make_loader(train_table, seed, training=True)
    validation_loader = make_loader(val_table, seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE,
        min_lr=MIN_LEARNING_RATE, threshold=0.0,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_rmse, best_epoch, stale_epochs = np.inf, 0, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loader.batch_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss_sum, epoch_n, accumulation_n = 0.0, 0, 0
        epoch_aggregate_sum, epoch_similarity_sum = 0.0, 0.0
        epoch_pair_count_sum, epoch_affinity_sum, epoch_zero_pair_chips = 0.0, 0.0, 0
        epoch_max_grad_norm = 0.0
        amp_skipped_updates = 0
        for step, (x, weights, y, _, similarity_valid) in enumerate(train_loader, start=1):
            x, weights, y = (value.to(DEVICE, non_blocking=True) for value in (x, weights, y))
            similarity_valid = similarity_valid.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                pred_map = model(x)
            predicted = aggregate_prediction(pred_map, weights)
            aggregate_loss_sum = F.smooth_l1_loss(predicted, y, beta=HUBER_BETA, reduction='sum')
            # Lambda zero follows the original backward path exactly; similarity is diagnostic only.
            with torch.set_grad_enabled(SIMILARITY_LAMBDA > 0):
                similarity_per_chip, pair_counts, mean_affinities = similarity_penalty_per_chip(
                    pred_map, x, similarity_valid, weights
                )
                similarity_loss_sum = similarity_per_chip.sum()
            loss_sum = (aggregate_loss_sum + SIMILARITY_LAMBDA * similarity_loss_sum
                        if SIMILARITY_LAMBDA > 0 else aggregate_loss_sum)
            if not torch.isfinite(similarity_loss_sum):
                raise FloatingPointError(f'Seed {seed}, epoch {epoch}, batch {step}: non-finite similarity')
            if not torch.isfinite(loss_sum):
                raise FloatingPointError(f'Seed {seed}, epoch {epoch}, batch {step}: non-finite loss')
            batch_n = len(y)
            scaler.scale(loss_sum).backward()
            accumulation_n += batch_n
            epoch_n += batch_n
            epoch_loss_sum += float(loss_sum.detach().cpu())
            epoch_aggregate_sum += float(aggregate_loss_sum.detach().cpu())
            epoch_similarity_sum += float(similarity_loss_sum.detach().cpu())
            epoch_pair_count_sum += float(pair_counts.sum().cpu())
            epoch_affinity_sum += float(mean_affinities.sum().cpu())
            epoch_zero_pair_chips += int((pair_counts == 0).sum().cpu())
            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                for parameter in model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.div_(accumulation_n)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM, error_if_nonfinite=False
                )
                # AMP can overflow on early steps; GradScaler skips that update and reduces its scale.
                if not torch.isfinite(grad_norm) and not AMP_ENABLED:
                    raise FloatingPointError('Non-finite gradients without AMP.')
                if torch.isfinite(grad_norm):
                    epoch_max_grad_norm = max(epoch_max_grad_norm, float(grad_norm))
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                amp_skipped_updates += int(AMP_ENABLED and scaler.get_scale() < previous_scale)
                optimizer.zero_grad(set_to_none=True)
                accumulation_n = 0

        validation_predictions = predict_table(model, validation_loader)
        metrics = regression_metrics(validation_predictions['observed_sif'], validation_predictions['predicted_sif_raw'])
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(metrics['rmse'])
        history.append({
            'seed': seed, 'epoch': epoch, 'train_loss': epoch_loss_sum / epoch_n,
            'train_aggregate_loss': epoch_aggregate_sum / epoch_n,
            'train_similarity_loss': epoch_similarity_sum / epoch_n,
            'train_weighted_similarity_loss': SIMILARITY_LAMBDA * epoch_similarity_sum / epoch_n,
            'similarity_lambda': SIMILARITY_LAMBDA,
            'similarity_pairs_per_chip': epoch_pair_count_sum / epoch_n,
            'similarity_mean_affinity': epoch_affinity_sum / epoch_n,
            'similarity_zero_pair_chip_fraction': epoch_zero_pair_chips / epoch_n,
            'val_rmse': metrics['rmse'], 'val_mae': metrics['mae'],
            'val_bias': metrics['bias'], 'val_r2': metrics['r2'],
            'learning_rate': current_lr, 'max_gradient_norm_before_clip': epoch_max_grad_norm,
            'amp_scale': float(scaler.get_scale()), 'amp_skipped_updates': amp_skipped_updates,
        })
        pd.DataFrame(history).to_csv(seed_dir / 'training_history.csv', index=False)
        print(f"seed={seed} epoch={epoch:03d} train={history[-1]['train_loss']:.5f} "
              f"aggregate={history[-1]['train_aggregate_loss']:.5f} "
              f"similarity={history[-1]['train_similarity_loss']:.5f} "
              f"weighted_similarity={history[-1]['train_weighted_similarity_loss']:.5f} "
              f"val_RMSE={metrics['rmse']:.5f} val_MAE={metrics['mae']:.5f} lr={current_lr:.2g}")
        if metrics['rmse'] < best_rmse:
            best_rmse, best_epoch, stale_epochs = metrics['rmse'], epoch, 0
            save_checkpoint(checkpoint_path, model, seed, epoch, best_rmse)
        else:
            stale_epochs += 1
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f'Seed {seed}: early stopping; best epoch {best_epoch}')
            break
    record = {
        'seed': int(seed), 'best_epoch': int(best_epoch), 'epochs_run': int(epoch),
        'best_validation_rmse': float(best_rmse), 'run_signature': RUN_SIGNATURE,
    }
    save_json(completed_path, record)
    del model, optimizer, scheduler, scaler, train_loader, validation_loader
    cleanup_memory()
    return record

training_records = [train_one_seed(seed) for seed in TRAINING_SEEDS]
training_summary = pd.DataFrame(training_records)
training_summary.to_csv(OUTPUT_DIR / 'training_seed_summary.csv', index=False)
display(training_summary.drop(columns='run_signature'))


## 9. Validation predictions and linear calibration
Fit observed = intercept + slope × raw_prediction on validation only, separately for each seed. The same transformation is then applied unchanged to test predictions and maps.

Calibrated validation scores describe the data used to fit calibration and are not independent estimates of calibration performance. The reference seed for detailed plots and permutation importance is chosen here using raw validation RMSE, before test evaluation. All three seeds are still reported.


In [ ]:
def fit_calibration(observed, predicted):
    observed = np.asarray(observed, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    if len(observed) < 2 or not np.isfinite(predicted).all() or predicted.std() <= 1e-12:
        raise ValueError('Validation predictions cannot support linear calibration.')
    design = np.column_stack([np.ones(len(predicted)), predicted])
    intercept, slope = np.linalg.lstsq(design, observed, rcond=None)[0]
    if not np.isfinite([intercept, slope]).all():
        raise ValueError('Non-finite calibration coefficients.')
    return {'intercept': float(intercept), 'slope': float(slope)}

def add_calibration(table, calibration):
    result = table.copy()
    result['predicted_sif_calibrated'] = (
        calibration['intercept'] + calibration['slope'] * result['predicted_sif_raw']
    )
    result['residual_calibrated'] = result['predicted_sif_calibrated'] - result['observed_sif']
    return result

calibrations, validation_predictions_by_seed = {}, {}
calibration_rows = []
for seed in TRAINING_SEEDS:
    model, checkpoint = load_seed_model(seed)
    loader = make_loader(val_table, seed)
    predictions = predict_table(model, loader)
    calibration = fit_calibration(predictions['observed_sif'], predictions['predicted_sif_raw'])
    calibrations[seed] = calibration
    predictions = add_calibration(predictions, calibration)
    predictions['seed'] = seed
    validation_predictions_by_seed[seed] = predictions
    seed_dir = OUTPUT_DIR / f'seed_{seed}'
    predictions.to_csv(seed_dir / 'validation_predictions.csv', index=False)
    save_json(seed_dir / 'calibration.json', calibration)
    checkpoint['calibration'] = calibration
    temporary = seed_dir / 'model_with_calibration.pt.tmp'
    torch.save(checkpoint, temporary)
    temporary.replace(seed_dir / 'model_with_calibration.pt')
    calibration_rows.append({
        'seed': seed, **calibration,
        'raw_validation_rmse': regression_metrics(predictions.observed_sif, predictions.predicted_sif_raw)['rmse'],
        'calibrated_validation_rmse_fitted': regression_metrics(predictions.observed_sif, predictions.predicted_sif_calibrated)['rmse'],
    })
    del model, checkpoint, loader
    cleanup_memory()
calibration_table = pd.DataFrame(calibration_rows)
calibration_table.to_csv(OUTPUT_DIR / 'validation_calibration_by_seed.csv', index=False)
REFERENCE_SEED = int(calibration_table.sort_values(['raw_validation_rmse', 'seed']).iloc[0]['seed'])
save_json(OUTPUT_DIR / 'reference_seed.json', {
    'seed': REFERENCE_SEED, 'selection': 'lowest raw validation RMSE; tie broken by seed',
})
print('Reference seed selected using validation only:', REFERENCE_SEED)
display(calibration_table)


## 10. Final evaluation of the frozen runs
Run this section after training settings are fixed. Every seed uses its own validation-fitted calibration. Training evaluation is unaugmented and is reported only as fitted performance; it is never pooled with held-out test results.

Mean and standard deviation across seeds quantify training randomness on this one fixed split, not uncertainty from alternative data splits or cross-validation.


In [ ]:
prediction_tables = {}
metric_rows = []
for seed in TRAINING_SEEDS:
    model, checkpoint = load_seed_model(seed)
    seed_predictions = {'validation': validation_predictions_by_seed[seed]}
    for split_name in ['train', 'test']:
        loader = make_loader(tables[split_name], seed)
        predictions = add_calibration(predict_table(model, loader), calibrations[seed])
        predictions['seed'] = seed
        seed_predictions[split_name] = predictions
        predictions.to_csv(OUTPUT_DIR / f'seed_{seed}' / f'{split_name}_predictions.csv', index=False)
        del loader
    prediction_tables[seed] = seed_predictions
    for split_name, predictions in seed_predictions.items():
        for version in ['raw', 'calibrated']:
            metric_rows.append({
                'seed': seed, 'split': split_name, 'prediction': version,
                'evaluation_scope': (
                    'held_out_test' if split_name == 'test' else
                    'validation_calibration_fit' if split_name == 'validation' and version == 'calibrated' else
                    'validation_model_selection' if split_name == 'validation' else 'training_fitted'
                ),
                **regression_metrics(predictions.observed_sif, predictions[f'predicted_sif_{version}']),
            })
    del model, checkpoint
    cleanup_memory()

seed_metrics = pd.DataFrame(metric_rows)
seed_metrics.to_csv(OUTPUT_DIR / 'metrics_by_seed_and_partition.csv', index=False)
summary_metrics = ['rmse', 'mae', 'bias', 'r2', 'slope', 'pearson_r', 'nrmse']
seed_summary = seed_metrics.groupby(['split', 'prediction'])[summary_metrics].agg(['mean', 'std'])
seed_summary.columns = ['_'.join(column) for column in seed_summary.columns]
seed_summary = seed_summary.reset_index()
seed_summary.to_csv(OUTPUT_DIR / 'metrics_mean_std_across_seeds.csv', index=False)
display(seed_metrics[seed_metrics['split'].eq('test')])
display(seed_summary[seed_summary['split'].eq('test')])
reference_test = prediction_tables[REFERENCE_SEED]['test'].copy()
reference_validation = prediction_tables[REFERENCE_SEED]['validation'].copy()


## 11. Diagnostics and validation permutation importance
All main error plots below use held-out test windows from the validation-selected reference seed. Subgroup tables also retain each of the other seeds separately. The count of distinct SIF dates is included because windows from one date are not independent replicates.


### 11.1 Training curves and residual behaviour
Training losses use standardized target units; RMSE/MAE use original SIF units, so they have separate axes. The loss-component figure separates aggregate Smooth L1, unweighted similarity and lambda-weighted similarity. Compare aggregate validation RMSE across lambda values, not the total training objective, whose definition changes with lambda. Pair-count and affinity diagnostics reveal whether the regularizer has usable support; a chip with no eligible pairs contributes zero to both the penalty and the logged per-chip mean affinity. Scatterplot slopes describe predicted versus observed SIF and are distinct from the validation calibration coefficients.


In [ ]:
histories = pd.concat([
    pd.read_csv(OUTPUT_DIR / f'seed_{seed}' / 'training_history.csv')
    for seed in TRAINING_SEEDS
], ignore_index=True)
histories.to_csv(OUTPUT_DIR / 'training_history_all_seeds.csv', index=False)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for seed, history in histories.groupby('seed'):
    for ax, column, title in zip(
        axes.ravel(), ['train_loss', 'val_rmse', 'val_mae', 'learning_rate'],
        ['Total training objective', 'Raw validation RMSE', 'Raw validation MAE', 'Learning rate'],
    ):
        ax.plot(history['epoch'], history[column], label=str(seed))
        ax.set(xlabel='Epoch', ylabel=title)
    best_epoch = int(training_summary.loc[training_summary.seed.eq(seed), 'best_epoch'].iloc[0])
    best_row = history.loc[history.epoch.eq(best_epoch)].iloc[0]
    axes[0, 1].scatter(best_epoch, best_row['val_rmse'], marker='*', s=130)
axes[1, 1].set_yscale('log')
axes[0, 0].legend(title='Seed')
save_figure(fig, 'training_curves')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
component_columns = [
    ('train_aggregate_loss', 'Aggregate Smooth L1'),
    ('train_similarity_loss', 'Unweighted similarity penalty'),
    ('train_weighted_similarity_loss', 'Lambda-weighted similarity penalty'),
    ('similarity_pairs_per_chip', 'Eligible neighbour pairs per chip'),
    ('similarity_mean_affinity', 'Mean pair affinity (averaged over chips)'),
    ('similarity_zero_pair_chip_fraction', 'Fraction of chips with no eligible pairs'),
]
for seed, history in histories.groupby('seed'):
    for ax, (column, title) in zip(axes.ravel(), component_columns):
        ax.plot(history['epoch'], history[column], label=str(seed))
        ax.set(xlabel='Epoch', ylabel=title)
axes[0, 0].legend(title='Seed')
fig.suptitle(f'Similarity regularization diagnostics | lambda={SIMILARITY_LAMBDA:g}')
fig.tight_layout()
save_figure(fig, 'training_loss_components_and_similarity_support')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, version in zip(axes, ['raw', 'calibrated']):
    observed = reference_test['observed_sif'].to_numpy()
    predicted = reference_test[f'predicted_sif_{version}'].to_numpy()
    metrics = regression_metrics(observed, predicted)
    low, high = min(observed.min(), predicted.min()), max(observed.max(), predicted.max())
    padding = max(0.03 * (high - low), 0.01)
    limits = (low - padding, high + padding)
    density = ax.hexbin(observed, predicted, gridsize=45, mincnt=1, bins='log', cmap='viridis')
    ax.plot(limits, limits, 'k--', linewidth=1, label='Identity')
    ax.plot(limits, metrics['intercept'] + metrics['slope'] * np.asarray(limits), color='crimson', label='Fitted line')
    ax.set(xlim=limits, ylim=limits, xlabel='Observed aggregate SIF', ylabel='Predicted aggregate SIF',
           title=f"{version.capitalize()} | RMSE={metrics['rmse']:.4f}, R²={metrics['r2']:.3f}\n"
                 f"slope={metrics['slope']:.3f}, n={metrics['n']}")
    ax.set_aspect('equal', adjustable='box')
    fig.colorbar(density, ax=ax, label='Windows per hexagon (log colour)')
axes[0].legend()
save_figure(fig, 'test_observed_predicted')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
residual_min = min(reference_test.residual_raw.min(), reference_test.residual_calibrated.min())
residual_max = max(reference_test.residual_raw.max(), reference_test.residual_calibrated.max())
edges = np.linspace(residual_min - 1e-6, residual_max + 1e-6, 45)
for version in ['raw', 'calibrated']:
    axes[0].hist(reference_test[f'residual_{version}'], bins=edges, alpha=0.45, label=version)
axes[0].axvline(0, color='black', linestyle='--')
axes[0].set(xlabel='Predicted minus observed SIF', ylabel='Windows', title='Test residual distributions')
axes[0].legend()
axes[1].scatter(reference_test.observed_sif, reference_test.residual_calibrated, s=10, alpha=0.35)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set(xlabel='Observed SIF', ylabel='Calibrated residual', title='Error versus target magnitude')
save_figure(fig, 'test_residual_diagnostics')


### 11.2 Subgroup performance and supervision support
SIF intervals include open-ended tails, so accepted values outside the old notebook's range are retained. Small groups are flagged; R² is omitted below MIN_GROUP_N or when the observed target variance is zero. Other metrics remain descriptive. No target-derived uncertainty weighting is used for training.


In [ ]:
SIF_EDGES = [-np.inf, -0.25, 0, 0.25, 0.5, 0.75, 1.0, 1.25, np.inf]
SIF_LABELS = ['<-0.25', '[-0.25,0)', '[0,0.25)', '[0.25,0.5)',
              '[0.5,0.75)', '[0.75,1)', '[1,1.25)', '>=1.25']
FOOTPRINT_EDGES = [3, 4, 6, 8, 10, 12, np.inf]
FOOTPRINT_LABELS = ['4', '5-6', '7-8', '9-10', '11-12', '13+']

def add_analysis_groups(table):
    result = table.copy()
    result['sif_bin'] = pd.cut(result.observed_sif, SIF_EDGES, labels=SIF_LABELS, right=False)
    result['footprint_bin'] = pd.cut(result.n_footprints, FOOTPRINT_EDGES, labels=FOOTPRINT_LABELS)
    return result

group_columns = ['month', 'sif_year', 'measurement_mode', 'mgrs_tile_t',
                 'land_cover_label', 'footprint_bin', 'sif_bin']
for optional in ['majority_BKR_NAME', 'states', 'hzs_values']:
    if optional in samples and samples[optional].fillna('').astype(str).str.strip().ne('').any():
        group_columns.append(optional)
subgroup_rows = []
for seed in TRAINING_SEEDS:
    test_predictions = add_analysis_groups(prediction_tables[seed]['test'])
    for column in group_columns:
        for label, group in test_predictions.groupby(column, observed=True, dropna=False):
            n_dates = group.Delta_Date.nunique()
            for version in ['raw', 'calibrated']:
                metrics = regression_metrics(group.observed_sif, group[f'predicted_sif_{version}'])
                if len(group) < MIN_GROUP_N:
                    metrics['r2'] = np.nan
                subgroup_rows.append({
                    'seed': seed, 'split': 'test', 'group_column': column,
                    'group_value': str(label), 'prediction': version,
                    'n_dates': n_dates,
                    'low_support': len(group) < MIN_GROUP_N or n_dates < MIN_GROUP_DATES,
                    **metrics,
                })
subgroup_metrics = pd.DataFrame(subgroup_rows)
subgroup_metrics.to_csv(OUTPUT_DIR / 'test_subgroup_metrics_all_seeds.csv', index=False)
reference_groups = subgroup_metrics[
    subgroup_metrics.seed.eq(REFERENCE_SEED) & subgroup_metrics.prediction.eq('calibrated')
].copy()
display(reference_groups)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, column, order in zip(axes, ['sif_bin', 'footprint_bin'], [SIF_LABELS, FOOTPRINT_LABELS]):
    group = reference_groups[reference_groups.group_column.eq(column)].set_index('group_value').reindex(order)
    colors = ['#d97706' if flag else '#2563eb' for flag in group.low_support.fillna(True)]
    ax.bar(np.arange(len(order)), group.rmse, color=colors)
    for position, (_, row) in enumerate(group.iterrows()):
        if pd.notna(row.rmse):
            ax.annotate(f"n={int(row.n)}", (position, row.rmse), ha='center', va='bottom', fontsize=8)
    ax.set(xticks=np.arange(len(order)), xticklabels=order, ylabel='Calibrated test RMSE', xlabel=column)
    ax.tick_params(axis='x', rotation=35)
fig.suptitle('Orange indicates low sample/date support')
save_figure(fig, 'test_error_by_target_and_footprint_count')

support_fields = [
    ('n_footprints', 'Contributing footprints'),
    ('aggregate_support_fraction', 'Fraction of chip covered by supervision'),
    ('mean_rasterized_mask_inside_fraction', 'Mean footprint fraction inside chip'),
    ('target_se', 'Within-window target SE (descriptive)'),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (column, label) in zip(axes.ravel(), support_fields):
    values = pd.to_numeric(reference_test[column], errors='coerce')
    valid = np.isfinite(values)
    ax.scatter(values[valid], np.abs(reference_test.loc[valid, 'residual_calibrated']), s=10, alpha=0.35)
    ax.set(xlabel=label, ylabel='Absolute calibrated test error')
save_figure(fig, 'test_error_vs_supervision_support')

# Seasonal and land-cover diagnostics remain test-only.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, column in zip(axes, ['month', 'land_cover_label']):
    group = reference_groups[reference_groups.group_column.eq(column)].copy()
    if column == 'month':
        group = group.assign(sort_key=pd.to_numeric(group.group_value)).sort_values('sort_key')
    else:
        group = group.sort_values('n', ascending=False)
    ax.bar(group.group_value, group.rmse)
    ax.set(xlabel=column, ylabel='Calibrated test RMSE')
    ax.tick_params(axis='x', rotation=45)
save_figure(fig, 'test_error_by_month_and_landcover')


### 11.3 Spatial residual distribution
Window centres are transformed using each row's stored window_crs, so both UTM zones can be represented correctly. This uses no external boundary files. Colours are clipped symmetrically at the 98th percentile for display only; saved residuals are unchanged.


In [ ]:
spatial_residuals = reference_test.copy()
spatial_residuals['longitude'] = np.nan
spatial_residuals['latitude'] = np.nan
for crs, group in spatial_residuals.groupby('window_crs'):
    transformer = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True)
    center_x = (group.cell_xmin.to_numpy() + group.cell_xmax.to_numpy()) / 2
    center_y = (group.cell_ymin.to_numpy() + group.cell_ymax.to_numpy()) / 2
    longitude, latitude = transformer.transform(center_x, center_y)
    spatial_residuals.loc[group.index, 'longitude'] = longitude
    spatial_residuals.loc[group.index, 'latitude'] = latitude
if not np.isfinite(spatial_residuals[['longitude', 'latitude']].to_numpy()).all():
    raise ValueError('Non-finite coordinates after CRS transformation.')
spatial_residuals.to_csv(OUTPUT_DIR / 'reference_test_spatial_residuals.csv', index=False)
colour_limit = max(float(np.quantile(np.abs(spatial_residuals.residual_calibrated), 0.98)), 1e-6)
fig, ax = plt.subplots(figsize=(8, 9))
points = ax.scatter(
    spatial_residuals.longitude, spatial_residuals.latitude,
    c=spatial_residuals.residual_calibrated, cmap='RdBu_r',
    vmin=-colour_limit, vmax=colour_limit, s=14, marker='s', linewidths=0,
)
ax.set(xlabel='Longitude', ylabel='Latitude',
       title=f'Calibrated test residuals | seed {REFERENCE_SEED} | n={len(spatial_residuals)}')
ax.set_aspect(1 / np.cos(np.deg2rad(spatial_residuals.latitude.mean())))
fig.colorbar(points, ax=ax, label='Predicted minus observed SIF')
save_figure(fig, 'test_spatial_residuals')


### 11.4 Representative predicted maps
Show examples near the 10th, 50th and 90th test-target percentiles plus the largest absolute test error. Selection is for inspection only. Dashed contours mark sampled footprint support; pixels outside it receive no direct target constraint for that window. No pixel-level truth is available.


In [ ]:
example_indices = []
for quantile in [0.1, 0.5, 0.9]:
    target_value = reference_test.observed_sif.quantile(quantile)
    index = int((reference_test.observed_sif - target_value).abs().idxmin())
    example_indices.append((index, f'target_quantile_{int(quantile * 100)}'))
example_indices.append((int(reference_test.residual_calibrated.abs().idxmax()), 'largest_absolute_error'))
model, checkpoint = load_seed_model(REFERENCE_SEED)
model.eval()
example_dataset = DensityDataset(test_table, augment=False)
example_records, used_indices = [], set()
for index, reason in example_indices:
    if index in used_indices:
        continue
    used_indices.add(index)
    row = reference_test.iloc[index]
    with np.load(row.shard_path, allow_pickle=False) as shard:
        raw_x = shard['X'][int(row.local_index)].astype(np.float32)
    x, weights, _, _ = example_dataset[index]
    with torch.no_grad():
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            normalized_map = model(x[None].to(DEVICE))[0, 0].float().cpu().numpy()
    raw_map = normalized_map * target_std + target_mean
    calibration = calibrations[REFERENCE_SEED]
    predicted_map = calibration['intercept'] + calibration['slope'] * raw_map
    weight_array = weights.numpy()
    map_aggregate = float(np.sum(predicted_map * weight_array, dtype=np.float64))
    record = {
        'aggregation_id': row.aggregation_id, 'reason': reason,
        'observed_sif': float(row.observed_sif),
        'predicted_sif_from_csv': float(row.predicted_sif_calibrated),
        'predicted_sif_from_example_map': map_aggregate,
        'map_min': float(predicted_map.min()), 'map_max': float(predicted_map.max()),
        'map_std': float(predicted_map.std()),
        'negative_pixel_fraction': float((predicted_map < 0).mean()),
    }
    example_records.append(record)
    panels = [
        (predicted_map, 'Calibrated predicted SIF', 'viridis', None, None),
        (raw_x[channel_names.index('ndvi')], 'NDVI', 'RdYlGn', -1, 1),
        (raw_x[channel_names.index('nirvp')], 'NIRvP', 'viridis', None, None),
        (raw_x[channel_names.index('winter_wheat_fraction')], 'Winter-wheat fraction', 'YlGn', 0, 1),
        (raw_x[channel_names.index('active_crop_fraction')], 'Active-crop fraction', 'YlGn', 0, 1),
        (weight_array, 'Equal-footprint supervision weights', 'magma', 0, None),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
    for ax, (array, title, cmap, vmin, vmax) in zip(axes.ravel(), panels):
        plot = ax.imshow(array, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(plot, ax=ax, fraction=0.046, pad=0.03)
    support = weight_array > 0
    if support.any() and not support.all():
        axes[0, 0].contour(support.astype(float), levels=[0.5], colors='white', linewidths=0.6, linestyles='--')
    fig.suptitle(f"{reason} | {row.aggregation_id}\nObserved={row.observed_sif:.4f}; "
                 f"map aggregate={map_aggregate:.4f}; footprints={int(row.n_footprints)}")
    save_figure(fig, f'example_map_{reason}')
pd.DataFrame(example_records).to_csv(OUTPUT_DIR / 'example_map_diagnostics.csv', index=False)
del model, checkpoint, example_dataset
cleanup_memory()


### 11.5 Validation permutation importance
Use the validation-selected reference seed with its calibration held fixed. Permute complete channel maps between validation windows, leaving target values and supervision weights unchanged. Repeat each permutation three times. An additional joint DOY permutation preserves the sine/cosine pairing.

A normalized validation cache is stored on disk using memory-mapped arrays (approximately 4.7 GB for 1,393 validation windows including weights). This avoids repeatedly decompressing NPZs for every channel/repeat. The cache is derived from the current inputs and verified by signature. Only one batch is copied to the GPU at a time.

The reported change in validation RMSE describes model dependence, not a causal effect. Correlated predictors and derived channels such as APAR/NIRvP can substitute for one another; independent permutation also breaks their normal relationships.


In [ ]:
permutation_results = pd.DataFrame()
if RUN_PERMUTATION_IMPORTANCE:
    import shutil

    cache_dir = OUTPUT_DIR / 'diagnostic_cache'
    cache_dir.mkdir(exist_ok=True)
    feature_path = cache_dir / 'validation_features.npy'
    weight_path = cache_dir / 'validation_weights.npy'
    cache_metadata_path = cache_dir / 'validation_cache.json'
    cache_signature = fingerprint({
        'normalization': NORMALIZATION_SIGNATURE,
        'validation_ids_in_order': val_table.aggregation_id.tolist(),
        'shape': [len(val_table), *CHIP_SHAPE], 'dtype': 'float32',
    })
    cache_ready = False
    if feature_path.exists() and weight_path.exists() and cache_metadata_path.exists():
        cache_ready = json.loads(cache_metadata_path.read_text()).get('signature') == cache_signature
    if not cache_ready:
        required_bytes = len(val_table) * (len(channel_names) + 1) * 200 * 200 * 4
        reusable_bytes = sum(path.stat().st_size for path in [feature_path, weight_path] if path.exists())
        if shutil.disk_usage(cache_dir).free + reusable_bytes < required_bytes * 1.10:
            raise RuntimeError('Insufficient disk space for validation permutation cache; free space or disable RUN_PERMUTATION_IMPORTANCE.')
        bank = np.lib.format.open_memmap(feature_path, mode='w+', dtype=np.float32, shape=(len(val_table), *CHIP_SHAPE))
        weight_bank = np.lib.format.open_memmap(weight_path, mode='w+', dtype=np.float32, shape=(len(val_table), 200, 200))
        loader = make_loader(val_table, REFERENCE_SEED)
        seen = np.zeros(len(val_table), dtype=int)
        for x, weights, _, indices in loader:
            indices = indices.numpy()
            bank[indices] = x.numpy()
            weight_bank[indices] = weights.numpy()
            seen[indices] += 1
        if not np.all(seen == 1):
            raise RuntimeError('Incomplete validation feature cache.')
        bank.flush()
        weight_bank.flush()
        del bank, weight_bank, loader
        save_json(cache_metadata_path, {'signature': cache_signature})
        cleanup_memory()

    bank = np.load(feature_path, mmap_mode='r', allow_pickle=False)
    weight_bank = np.load(weight_path, mmap_mode='r', allow_pickle=False)
    if bank.shape != (len(val_table), *CHIP_SHAPE) or weight_bank.shape != (len(val_table), 200, 200):
        raise ValueError('Validation cache shape mismatch.')
    model, checkpoint = load_seed_model(REFERENCE_SEED)
    model.eval()

    @torch.no_grad()
    def predict_permuted(feature_indices=(), donors=None):
        predictions = np.empty(len(val_table), dtype=np.float64)
        for start in range(0, len(val_table), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(val_table))
            batch = np.array(bank[start:stop], copy=True)
            if donors is not None:
                for feature_index in feature_indices:
                    batch[:, feature_index] = bank[donors[start:stop], feature_index, :, :]
            x = torch.from_numpy(batch).to(DEVICE)
            weights = torch.from_numpy(np.array(weight_bank[start:stop], copy=True)).to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                pred_map = model(x)
            raw = aggregate_prediction(pred_map, weights).cpu().numpy() * target_std + target_mean
            calibration = calibrations[REFERENCE_SEED]
            predictions[start:stop] = calibration['intercept'] + calibration['slope'] * raw
        return predictions

    observed = val_table.y_aggregate.to_numpy(dtype=np.float64)
    baseline_prediction = predict_permuted()
    baseline_rmse = regression_metrics(observed, baseline_prediction)['rmse']
    groups_to_permute = [(name, [i]) for i, name in enumerate(channel_names)]
    groups_to_permute.append(('doy_sin_and_cos_joint', [channel_names.index('doy_sin'), channel_names.index('doy_cos')]))
    donors_by_repeat = [
        np.random.default_rng(SPLIT_SEED + 1000 + repeat).permutation(len(val_table))
        for repeat in range(PERMUTATION_REPEATS)
    ]
    rows = []
    for name, feature_indices in groups_to_permute:
        for repeat, donors in enumerate(donors_by_repeat):
            predictions = predict_permuted(feature_indices, donors)
            rmse = regression_metrics(observed, predictions)['rmse']
            rows.append({
                'reference_seed': REFERENCE_SEED, 'feature': name, 'repeat': repeat + 1,
                'baseline_calibrated_validation_rmse': baseline_rmse,
                'permuted_calibrated_validation_rmse': rmse, 'delta_rmse': rmse - baseline_rmse,
            })
        pd.DataFrame(rows).to_csv(OUTPUT_DIR / 'validation_permutation_repeats.csv', index=False)
        print('Permutation completed:', name)
    permutation_results = pd.DataFrame(rows).groupby('feature')['delta_rmse'].agg(['mean', 'std']).reset_index()
    permutation_results = permutation_results.sort_values('mean', ascending=True)
    permutation_results.to_csv(OUTPUT_DIR / 'validation_permutation_summary.csv', index=False)
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.barh(permutation_results.feature, permutation_results['mean'], xerr=permutation_results['std'].fillna(0), color='#2563eb')
    ax.axvline(0, color='black', linewidth=1)
    ax.set(xlabel='Increase in calibrated validation RMSE', ylabel='Permuted channel(s)',
           title=f'Validation permutation importance | seed {REFERENCE_SEED} | {PERMUTATION_REPEATS} repeats')
    save_figure(fig, 'validation_permutation_importance')
    del bank, weight_bank, model, checkpoint
    cleanup_memory()


## 12. Saved results and interpretation
Each seed directory contains training history (including separate aggregate, similarity and total losses), its best raw checkpoint, a self-contained checkpoint with validation calibration, and train/validation/test predictions. Similarity settings are included in experiment_config.json, checkpoint training_config and the run signature. The experiment directory contains the fixed split, all-training normalization, seed summaries, test subgroup metrics, figures and validation permutation results.

Use model_with_calibration.pt for subsequent inference with exactly the stored channel order, predictor normalization, target scaling and DOY/NIRvP definitions. Monthly crop activity still follows the preparation script's crop calendar.

Do not interpret the three-seed standard deviation as a confidence interval, or fitted/validation-calibration metrics as independent test results. No combined train/validation/test performance is used as the main result.


In [ ]:
save_json(OUTPUT_DIR / 'results_manifest.json', {
    'run_name': RUN_NAME, 'run_signature': RUN_SIGNATURE,
    'dataset_fingerprint': DATASET_FINGERPRINT,
    'n_windows': int(len(samples)), 'n_dates': int(samples.Delta_Date.nunique()),
    'partition_counts': {name: int(len(table)) for name, table in tables.items()},
    'reference_seed': REFERENCE_SEED,
    'reference_selection': 'raw validation RMSE only',
    'training_seeds': TRAINING_SEEDS,
    'split_file': 'chip_splits.csv',
    'normalization_file': 'normalization_stats.npz',
    'metrics_file': 'metrics_by_seed_and_partition.csv',
    'seed_summary_file': 'metrics_mean_std_across_seeds.csv',
    'subgroup_file': 'test_subgroup_metrics_all_seeds.csv',
    'inference_checkpoints': [f'seed_{seed}/model_with_calibration.pt' for seed in TRAINING_SEEDS],
    'permutation_completed': bool(not permutation_results.empty),
    'evaluation_scope': 'held-out SIF dates within sampled regions',
    'sentinel_source_independence_enforced': False,
    'cross_validation_performed': False, 'unseen_tile_evaluation_performed': False,
})
print('Saved results to:', OUTPUT_DIR)
print('Inference checkpoints:')
for seed in TRAINING_SEEDS:
    print(OUTPUT_DIR / f'seed_{seed}' / 'model_with_calibration.pt')
display(seed_summary[seed_summary['split'].eq('test')])
